In [0]:
%run ../utils

In [0]:
# Explode emails array from leads_processed_contacts into leads_processed_contacts_emails
# Schema discovered dynamically — no hard-coded field names from the struct.
# "email_data.*" expands ALL fields whatever they are.

from pyspark.sql.functions import col, explode_outer, md5, concat_ws
import time

start = time.time()

print(f"{'='*80}")
print(f"📥 leads_processed_contacts → leads_processed_contacts_emails")
print(f"{'='*80}")

# CDC watermark: last write time of the emails table
watermark = get_silver_watermark('leads_processed_contacts_emails')
if watermark:
    print(f"  CDC MODE: reading changes after {watermark}")
else:
    print(f"  FULL LOAD: first run")

# Read from contacts silver table — only rows newer than the emails watermark
df_contacts = spark.table("crm_ingestion.silver.leads_processed_contacts")
if watermark:
    df_contacts = df_contacts.filter(col("bronze_updated_at") > watermark)

# Explode the emails array — one row per email per contact
# "email_data.*" dynamically expands ALL struct fields (no hard-coding)
df_emails = (
    df_contacts
    .select(
        col("leads_processed_id"),
        col("id").alias("leads_processed_contacts_id"),
        col("bronze_insert_date"),
        col("bronze_updated_at"),
        explode_outer(col("emails")).alias("email_data")
    )
    .select(
        "leads_processed_id",
        "leads_processed_contacts_id",
        "bronze_insert_date",
        "bronze_updated_at",
        "email_data.*"  # Discover all struct fields dynamically
    )
)

# Add record_hash for deduplication (hash all discovered columns)
hash_cols = [col(c) for c in df_emails.columns]
df_emails = df_emails.withColumn("record_hash", md5(concat_ws("|", *hash_cols)))

# Write to silver table — CDC: delete rows newer than the watermark, then append.
# First run (watermark=None) is a full overwrite.
row_count = df_emails.count()
if row_count == 0:
    print(f"  ✅ CDC: No new rows since {watermark}")
else:
    _CDC_WATERMARK = watermark  # activates write_silver's delete+append path
    write_silver(df_emails, "leads_processed_contacts_emails")
    _CDC_WATERMARK = None
    print(f"\n✅ Table updated: crm_ingestion.silver.leads_processed_contacts_emails")
    print(f"🔢 Rows this batch: {row_count:,}")
    print(f"📊 Total rows: {spark.table('crm_ingestion.silver.leads_processed_contacts_emails').count():,}")
print(f"\n📋 SCHEMA:")
spark.table("crm_ingestion.silver.leads_processed_contacts_emails").printSchema()
print(f"\n📊 SAMPLE ROWS:")
display(spark.table("crm_ingestion.silver.leads_processed_contacts_emails").limit(10))
print(f"\n⏱️ Completed in {time.time() - start:.1f}s")
print(f"{'='*80}")